# ex03 · nb03 — 인터랙티브 실시간 놀이터 🎮

이 노트북은 **"고쳐서 바로 던져보는"** 개발 루프 그 자체가 주제야. 코드 예제라기보다,
**MoveItPy 를 켜둔 커널을 놀이터 삼아 팔을 실시간으로 쿡쿡 찔러보는** 워크플로우 시연이지.

## 왜 이게 그렇게 강력하냐면

기존 루프는 이랬어:

> **파일 고침 → `Ctrl-C` → `ros2 launch ...` → 30초 대기 → 로그 구경 → 아 오타났네 → 다시 처음부터**

이 노트북에선 그 루프가 이렇게 **붕괴**해:

> **셀 고침 → `Shift+Enter` → 즉시 팔이 움직임** ✨

핵심은 **커널이 살아있다**는 거야. 아래 설정 셀을 한 번 실행하면 `MoveItPy` 객체가
커널 메모리에 상주하고, 그 뒤로:

- `move_group` 도, 컨트롤러도 **절대 안 죽어**. 세션 내내 그대로 붙어 있음.
- 셀을 고쳐서 `Shift+Enter` 만 하면 **그 순간** 목표가 계획·실행돼.
- **`colcon` 재빌드 없음. `ros2 launch` 재시작 없음.** 파이썬 코드는 커널 안에서 즉석 재정의되니까.

> ⚠️ **재시작이 필요한 유일한 경우**: URDF/SRDF/컨트롤러(ros2_control) 같은 **로봇 정의 자체**를
> 바꿨을 때뿐이야. 그건 런치가 노드에 주입하는 파라미터라 커널만으론 못 바꿔. 그 외 순수 파이썬 로직은
> 전부 셀에서 즉석으로 갈아끼울 수 있어.

## 시작 전 체크 ✅

- 이 노트북은 **런치 스택이 먼저 떠 있어야** 돌아가. 별도 터미널에서
  `ros2 launch ... ex03_moveit_py_notebook.launch.py` 를 띄우고 **약 30초** 기다린 뒤 (move_group/컨트롤러 준비) 아래 설정 셀을 실행해.
- 🔒 **설정 셀(다음 두 셀)은 커널당 딱 한 번만** 실행할 것. `MoveItPy` 는 커널당 하나만 살 수 있어.
  두 번 실행하면 노드 이름 충돌로 꼬여. 다시 처음부터 하고 싶으면 **커널을 재시작**하고 한 번만 실행해.
- 🔒 한 번에 **노트북/커널 하나만** 띄워. 동시에 두 개 커널이 `MoveItPy` 를 잡으면 서로 싸워.


In [ ]:
import time
import numpy as np
from geometry_msgs.msg import PoseStamped
from moveit.core.kinematic_constraints import construct_joint_constraint
from moveit.core.robot_state import RobotState
from moveit.planning import MoveItPy, MultiPipelinePlanRequestParameters
from rclpy.logging import get_logger
import rclpy
from moveit_py_params import build_moveit_config_dict

if not rclpy.ok():
    rclpy.init()

logger = get_logger("nb03_playground")
robot = MoveItPy(node_name="moveit_py", config_dict=build_moveit_config_dict())
arm = robot.get_planning_component("arm")
gripper = robot.get_planning_component("gripper")
robot_model = robot.get_robot_model()
print("놀이터 준비 완료 — 아래 헬퍼 셀 실행 후, 마지막 섹션에서 값만 바꿔가며 Shift+Enter")

## 🧰 헬퍼 함수 — 이 노트북의 심장

여기가 진짜 핵심이야. 짧고 재사용 가능한 함수 몇 개만 커널에 등록해두면, 그 다음부터는
`go_named("home")`, `go_joints(joint1=0.5)`, `grip("gripper_open")` 같은 **한 줄**로 팔을 조종해.

패턴은 전부 동일해: **목표 세팅 → `plan()` → 되면 `execute()`**. 이 지겨운 3단계를 `_run()`
하나로 감싸버려서, 위쪽 함수들은 "무엇을" 만 신경 쓰고 "어떻게" 는 신경 안 써도 되게 했어.

> 💡 이 셀도 그냥 파이썬 함수 정의라, 나중에 함수 동작이 맘에 안 들면 **여기서 고치고 `Shift+Enter`**
> → 커널 안 정의가 즉석 교체돼. 아래 놀이터 셀들은 자동으로 새 정의를 쓰게 돼. 이게 REPL 개발의 맛이지.

In [ ]:
def _run(component, sleep_time=0.5):
    """목표가 세팅된 planning_component 를 계획→실행. 성공 여부(bool) 반환."""
    result = component.plan()
    if result:
        robot.execute(result.trajectory, controllers=[])  # controllers=[] → 자동 선택
    else:
        logger.error("계획 실패 (도달 불가/충돌?)")
    time.sleep(sleep_time)
    return bool(result)


def go_named(name, group=None):
    "SRDF 이름 상태로 이동. group 기본은 arm. (arm: home / gripper: gripper_open 등)"
    comp = group or arm
    comp.set_start_state_to_current_state()
    comp.set_goal_state(configuration_name=name)
    return _run(comp)


def go_joints(**joints):
    "관절 목표로 이동. 예: go_joints(joint1=0.5, joint2=0.3). 안 준 관절은 0.0 으로 채움."
    arm.set_start_state_to_current_state()
    gs = RobotState(robot_model)
    # RobotState 는 모든 관절이 기본 0.0 으로 정의돼 있음. 안전하게 arm 6관절을
    # 명시적으로 0.0 으로 깔고, 사용자가 준 값만 덮어써서 부분 dict 도 예측 가능하게.
    positions = {f"joint{i}": 0.0 for i in range(1, 7)}
    positions.update({k: float(v) for k, v in joints.items()})
    gs.joint_positions = positions
    jc = construct_joint_constraint(
        robot_state=gs,
        joint_model_group=robot_model.get_joint_model_group("arm"),
    )
    arm.set_goal_state(motion_plan_constraints=[jc])
    return _run(arm)


def grip(name):
    "그리퍼 이름 상태: gripper_open / gripper_half / gripper_close"
    gripper.set_start_state_to_current_state()
    gripper.set_goal_state(configuration_name=name)
    return _run(gripper)


print("헬퍼 등록 완료: go_named / go_joints / grip  (그리고 내부용 _run)")

## 🔎 현재 상태 들여다보기 (introspection)

인터랙티브 개발의 절반은 **"지금 로봇이 어디 있냐"** 를 즉석에서 확인하는 거야. 계획을 보내기 전에
현재 관절값을 찍어보고, 목표를 보낸 뒤엔 tcp_link 가 실제로 어디로 갔는지 FK(순기구학)로 확인해.

아래 셀은 `planning_scene_monitor` 의 **읽기 전용 스냅샷**을 떠서:

- 6개 관절의 현재 각도(rad)
- `tcp_link` 의 현재 월드 위치(x, y, z, base_link 기준)

를 출력해. 팔을 움직인 **직후에 이 셀을 다시 `Shift+Enter`** 하면 결과가 어떻게 바뀌는지 바로 보여.
이게 놀이터에서 "찔러보고 → 확인하고 → 또 찔러보는" 리듬이야.

In [ ]:
# 현재 상태 스냅샷 — 움직인 뒤 다시 실행하면 값이 바뀐다.
try:
    psm = robot.get_planning_scene_monitor()
    with psm.read_only() as scene:
        st = scene.current_state
        st.update()  # FK 등 파생값 갱신
        cur = {j: round(st.joint_positions[j], 3)
               for j in ["joint1", "joint2", "joint3", "joint4", "joint5", "joint6"]}
        print("현재 관절:", cur)
        tf = st.get_global_link_transform("tcp_link")  # 4x4 동차변환
        print("tcp_link 위치:", np.round(np.asarray(tf)[:3, 3], 4))
except Exception as exc:
    print(f"상태 조회 실패({type(exc).__name__}: {exc})")
    print("→ 런치 스택/컨트롤러가 다 떴는지, 설정 셀을 실행했는지 확인.")

## 🕹️ 라이브 포킹 — **여기부터가 핵심**

여기부터가 이 노트북의 진짜 놀이터야. **아래 셀들은 전부 한 줄짜리**고, 값만 바꿔서
`Shift+Enter` 하면 **즉시 팔이 움직여**. `colcon` 재빌드도, 런치 재시작도 없음.

숫자를 조금씩 바꿔가며 감을 잡아봐. 관절 크기는 대략 **±1.0 rad 이내**로 유지하면 리미트 안 넘고 안전해.
움직인 뒤엔 위의 🔎 introspection 셀을 다시 실행해서 어디로 갔는지 확인하는 습관을 들여.

**1) 홈 자세로** — 언제든 여기로 돌아오면 기준점.

In [ ]:
go_named("home")

**2) 오른쪽으로 살짝** — joint1 을 +로.

In [ ]:
go_joints(joint1=0.6, joint2=0.4, joint3=-0.5)

**3) 왼쪽으로 살짝** — joint1 부호만 뒤집음. 숫자 하나 바꿔서 대칭 확인.

In [ ]:
go_joints(joint1=-0.6, joint2=0.4, joint3=-0.5)

**4) 그리퍼 열기** 👐

In [ ]:
grip("gripper_open")

**5) 그리퍼 닫기** ✊ — 중간은 `grip("gripper_half")`.

In [ ]:
grip("gripper_close")

**6) 팔꿈치+손목만 살짝** — 안 준 관절은 헬퍼가 0.0 으로 채우니 이 두 개만 신경 쓰면 됨.

In [ ]:
go_joints(joint2=0.5, joint3=-0.7, joint5=0.3)

**7) 자유 실험 칸** — 여기다 아무 값이나 넣고 굴려봐. (예시는 아주 작은 움직임)

In [ ]:
go_joints(joint1=0.2, joint4=0.3, joint6=0.4)

## 🧠 다중 파이프라인 계획 (multi-pipeline)

MoveIt2 는 한 목표에 대해 **여러 플래너를 동시에 돌리고 그중 되는(또는 제일 좋은) 걸 채택**할 수 있어.
여기선 런치/params 에 정의된 두 파이프라인을 같이 던져:

- `ompl_rrtc` — OMPL RRTConnect (샘플링 기반, 뭐든 잘 푸는 범용 플래너)
- `pilz_ptp` — Pilz PTP (관절공간 직선 보간, 산업용 예측 가능한 궤적)

둘을 동시에 계획하면, 하나가 막혀도 다른 하나가 답을 낼 확률이 올라가. 아래 셀은 리포 ex03 과 동일하게
**다중 파이프라인 params 가 없으면 단일 계획으로 폴백**하도록 `try/except` 로 감쌌어 (환경에 따라
params 미로딩이면 조용히 일반 `plan()` 으로 떨어짐).

In [ ]:
arm.set_start_state_to_current_state()
arm.set_goal_state(configuration_name="home")
try:
    multi = MultiPipelinePlanRequestParameters(robot, ["ompl_rrtc", "pilz_ptp"])
    result = arm.plan(multi_plan_parameters=multi)
except Exception as exc:  # 파이프라인 params 미설정 시 단일 계획으로 폴백
    logger.warn(f"다중 파이프라인 불가({exc}) — 단일 계획 폴백")
    arm.set_start_state_to_current_state()
    arm.set_goal_state(configuration_name="home")
    result = arm.plan()
if result:
    robot.execute(result.trajectory, controllers=[])
    print("실행 완료")
else:
    logger.error("계획 실패")

## 🎬 나만의 시퀀스 만들기

한 줄씩 찔러보는 게 익숙해졌으면, 이제 **리스트로 엮어서** 동작을 스크립트처럼 돌려볼 수 있어.
아래는 목표들의 리스트를 순회하면서 순서대로 실행하는 아주 작은 예시야. `steps` 리스트만 고치면
너만의 안무를 커널 안에서 즉석으로 만들 수 있어 — 역시 재빌드/재시작 없이.

In [ ]:
# ("종류", 값) 튜플 리스트를 순서대로 실행. 값만 바꿔서 나만의 안무를 짜봐.
steps = [
    ("named", "home"),
    ("grip", "gripper_open"),
    ("joints", {"joint1": 0.4, "joint2": 0.3}),
    ("grip", "gripper_close"),
    ("joints", {"joint1": -0.4, "joint2": 0.3}),
    ("named", "home"),
]

for i, (kind, val) in enumerate(steps, 1):
    print(f"[{i}/{len(steps)}] {kind}: {val}")
    if kind == "named":
        ok = go_named(val)
    elif kind == "joints":
        ok = go_joints(**val)
    elif kind == "grip":
        ok = grip(val)
    else:
        print(f"  알 수 없는 종류: {kind}")
        continue
    if not ok:
        print("  → 실패, 시퀀스 중단")
        break
print("시퀀스 끝")

## 🏁 정리 — 개발 루프의 승리

- **셀 고침 → `Shift+Enter` → 즉시.** 커널이 살아있는 한 `move_group`/컨트롤러는 안 죽고,
  파이썬 로직은 셀에서 즉석 재정의돼. `colcon` 재빌드도 `ros2 launch` 재시작도 없어. 이게 이 노트북의 전부야.
- 🔒 **설정 셀(맨 위 두 셀)은 다시 실행하지 마.** 헬퍼/놀이터 셀은 얼마든지 재실행해도 되지만,
  `MoveItPy` 생성 셀은 커널당 한 번뿐. 새로 시작하려면 커널 재시작.
- 💀 커널을 끌 때 뜨는 **"Kernel died"** 는 MoveItPy 종료자(destructor)의 **알려진 양성 이슈**야.
  작업은 이미 다 끝난 뒤라 무시해도 돼.

### 실물 로봇으로 넘어갈 때 ⚠️

여기 있는 함수(`go_named`/`go_joints`/`grip`)는 **실물 Piper 에서도 그대로 동작**해 —
mock 대신 실물 스택을 런치하면 끝이야. 단, 실물은 **속도 스케일링을 낮게** 두고 팔에서 눈을 떼지 말 것,
그리고 첫 동작 전엔 반드시 **리포의 실물 로봇 체크리스트**를 따를 것. 이 노트북은 mock 기준 놀이터라
안전 절차는 거기서 다룬다.